In [2]:
import biom
import numpy as np
import pandas as pd
from biom.util import biom_open
import re

In [7]:
birch_metadata = pd.read_csv("/home/jokirkland/scripts/knight_lab_tools/jupyter_notebooks/16188_20260928_cleaned_study-specific.tsv", sep="\t")

derived_categories_for_metadata = pd.read_csv("/home/eclift/birch-data/derived_categories_for_metadata_birch_aquarium.tsv", sep="\t")
derived_categories_for_metadata = derived_categories_for_metadata.drop(columns='sample_name')

biom_table = biom.load_table("/home/jokirkland/scripts/knight_lab_tools/jupyter_notebooks/sylph_merged_species_fixed_birch_aquarium.biom")
biom_table2 = biom.load_table("/home/eclift/birch-data/mindaroo_y1_report_602_samples_20260928_cell_counts_per_liter_combined_birch_aquarium.biom")

# tank data
long_tank_metadata = pd.read_csv("/home/eclift/birch-data/birch_edna_tank_organisms_long.tsv", sep="\t")
tank_metadata = pd.read_csv("/home/eclift/birch-data/birch_edna_unique_organisms.tsv", sep="\t")

In [21]:
# merged_df comes from birch_metadata and derived_categories_for_metadata
merged_df = pd.merge(birch_metadata, derived_categories_for_metadata, on='anonymized_name', how='outer')
merged_df = merged_df.rename(columns={'anonymized_name': 'id', 'sample_name':'full_sample_name'})
new_order = ["id"] + [col for col in merged_df.columns if col !="id"]
merged_df = merged_df[new_order]

meta = merged_df.copy()
meta = meta.replace({'not applicable': np.nan, 'not collected': np.nan, 'missing': np.nan})

biom_ids = set(biom_table.ids())
print(biom_ids)

meta['md_id'] = meta['id'] 

def norm(x):
    return re.sub(r'[^a-z0-9]+', '_', str(x).lower()).strip('_')

# adding tank data from birch_edna_unique_organisms_long (long_tank_metadata)
long_df = long_tank_metadata.copy()
long_df['tank_key'] = long_df['tank'].map(norm)
tank_keys = set(long_df['tank_key']) | set(long_df['zims_enclosure_name'].map(norm))

meta['tank_key'] = meta['tank'].map(lambda x: norm(x) if pd.notna(x) else np.nan)
print(sorted(set(meta['tank_key'].dropna()) - set(long_df['tank_key'])))

long_df['genus'] = long_df['scientific_name'].str.split().str[0]

tank_feat = (long_df.groupby('tank_key')
    .agg(zims_enclosure_name=('zims_enclosure_name', 'first'),
         source_sheet=('source_sheet', 'first'),
         n_species=('scientific_name', 'nunique'),
         n_genera=('genus', 'nunique'),
         n_individuals_start=('count_beginning', 'sum'),
         n_individuals_end=('count_ending', 'sum'),
         species_list=('scientific_name', lambda s: '; '.join(sorted(set(s)))))
    .reset_index())

print(tank_feat)

# write full metadata file
meta_full = (meta.merge(tank_feat, on='tank_key', how='left', validate='m:1'))
meta_full.to_csv('metadata_full.tsv', sep='\t', index=False)

{np.str_('MD0951'), np.str_('MD0974'), np.str_('MD0978'), np.str_('MD0985'), np.str_('MD0989'), np.str_('MD1001'), np.str_('MD0990'), np.str_('MD0949'), np.str_('MD0993'), np.str_('MD0999'), np.str_('MD0943'), np.str_('MD0958'), np.str_('MD0948'), np.str_('MD0984'), np.str_('MD0954'), np.str_('MD0976'), np.str_('MD0941'), np.str_('MD0971'), np.str_('MD0987'), np.str_('MD0961'), np.str_('MD0953'), np.str_('MD0977'), np.str_('MD0947'), np.str_('MD0998'), np.str_('MD1000'), np.str_('MD0967'), np.str_('MD0959'), np.str_('MD0996'), np.str_('MD0994'), np.str_('MD0962'), np.str_('MD0975'), np.str_('MD0966'), np.str_('MD0952'), np.str_('MD0945'), np.str_('MD0997'), np.str_('MD0979'), np.str_('MD0982'), np.str_('MD0970'), np.str_('MD0956'), np.str_('MD0960'), np.str_('MD0980'), np.str_('MD0942'), np.str_('MD0981'), np.str_('MD0969'), np.str_('MD0965'), np.str_('MD0968'), np.str_('MD0944'), np.str_('MD0988'), np.str_('MD0991'), np.str_('MD0995'), np.str_('MD0964'), np.str_('MD0986'), np.str_('MD

/tmp/ipykernel_1516095/2748524242.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  meta = meta.replace({'not applicable': np.nan, 'not collected': np.nan, 'missing': np.nan})


In [9]:
# replace '_other.fastq.gz' in sylph biom tbl
sylph_biom_ids = biom_table.ids(axis='sample')
sylph_biom_ids_dict = {name: name.replace('_other.fastq.gz', '') for name in sylph_biom_ids}
biom_table = biom_table.update_ids(sylph_biom_ids_dict, axis='sample')
sylph_biom_ids = biom_table.ids(axis='sample')

sylph_biom_ids = biom_table.ids(axis='sample') 

metadata_filtered = meta_full[meta_full['md_id'].notna()].copy()
metadata_filtered = metadata_filtered.rename(columns={'id': 'SampleID'})

sylph_metadata_ids = set(metadata_filtered['SampleID'])
sylph_intersection = list(set(sylph_biom_ids).intersection(sylph_metadata_ids))
sylph_filtered_biom = biom_table.filter(sylph_intersection, axis='sample', inplace=False)
sylph_filtered_biom.remove_empty()      

metadata_filtered = metadata_filtered[metadata_filtered['SampleID'].isin(set(sylph_filtered_biom.ids(axis='sample')))].copy()

# check regex again later
metadata_filtered['tank'] = metadata_filtered['tank'].str.replace(r'\s+', '_', regex=True)

print(metadata_filtered.head())

    SampleID full_sample_name bottle_number cast_number chlorophyll_max  \
789   MD0941      16188.MD941           NaN         NaN             NaN   
790   MD0942      16188.MD942           NaN         NaN             NaN   
791   MD0943      16188.MD943           NaN         NaN             NaN   
792   MD0944      16188.MD944           NaN         NaN             NaN   
793   MD0945      16188.MD945           NaN         NaN             NaN   

    collection_date collection_timestamp comment country depth_m  ...  \
789          7/1/26         7/1/26 10:00     NaN     USA     NaN  ...   
790          7/1/26         7/1/26 10:00     NaN     USA     NaN  ...   
791          7/1/26         7/1/26 10:00     NaN     USA     NaN  ...   
792          7/1/26         7/1/26 10:00     NaN     USA     NaN  ...   
793          7/1/26         7/1/26 10:00     NaN     USA     NaN  ...   

    genus_Turbo genus_Urobatis genus_Urticina genus_Xenistius  \
789         0.0            0.0            0.0

In [22]:

assert len(meta_full) == len(birch_metadata), (len(meta_full), len(birch_metadata))
assert meta_full['id'].is_unique
assert not [c for c in meta_full.columns if c.endswith(('_x', '_y'))]

birch = meta_full[meta_full['derived_site'] == 'Birch Aquarium']

print("Birch samples:", len(birch))
print("with tank_key:", birch['tank_key'].notna().sum())
print("with tank features (n_species):", birch['n_species'].notna().sum())

# samples that have a tank name but no organism data = failed matches
missing = birch[birch['tank_key'].notna() & birch['n_species'].isna()]
print(missing['tank_key'].value_counts())

sample_tanks = set(birch['tank_key'].dropna())
organism_tanks = set(long_df['tank_key'])

print("organism tanks with no sample:", sorted(organism_tanks - sample_tanks))
print("sample tanks with no organisms:", sorted(sample_tanks - organism_tanks))

for t in birch['tank_key'].dropna().unique():
    expected = long_df.loc[long_df['tank_key'] == t, 'scientific_name'].nunique()
    got = meta_full.loc[meta_full['tank_key'] == t, 'n_species'].dropna().unique()
    if expected == 0:
        assert len(got) == 0, (t, got)          # should be NaN
        print(t, "no organism data (expected)")
    else:
        assert len(got) == 1 and got[0] == expected, (t, expected, got)
        print(t, "ok", expected)

# species per file
long_species = set(long_df['scientific_name'])
uniq_species = set(tank_metadata['scientific_name'])
print("only in long:", sorted(long_species - uniq_species)[:10])
print("only in unique:", sorted(uniq_species - long_species)[:10])

# total individuals
print(long_df['count_ending'].sum(), tank_metadata['total_individuals'].sum())

# number of tanks per species
long_ntanks = long_df.groupby('scientific_name')['tank_key'].nunique()
cmp = tank_metadata.set_index('scientific_name')['n_tanks'].to_frame().join(long_ntanks)
print(cmp[cmp['n_tanks'] != cmp['tank_key']])


m = meta_full.set_index('md_id')
shared = m.index.intersection(sylph_filtered_biom.ids('sample'))
print(len(shared), "of", sylph_filtered_biom.shape[1])
print(m.loc[shared, ['tank_key', 'n_species']].isna().sum())
print(m.loc[shared, 'tank_key'].value_counts())   # samples per tank
print(sorted(long_df['tank_key'].unique()))


Birch samples: 61
with tank_key: 61
with tank features (n_species): 57
tank_key
wrd        1
or_6       1
bts_gpo    1
sd_3       1
Name: count, dtype: int64
organism tanks with no sample: []
sample tanks with no organisms: ['bts_gpo', 'or_6', 'sd_3', 'wrd']
wra ok 7
wrb ok 3
wrc ok 25
wrd no organism data (expected)
wre ok 1
wrf ok 3
wrg ok 1
wrh ok 5
wri ok 4
mangrove ok 6
tank_24 ok 4
coral ok 67
open_ocean ok 6
hex_1 ok 7
hex_2 ok 12
tank_23 ok 7
mpa_touch ok 10
center_tidepool ok 10
tank_22 ok 25
north_tidepool ok 11
south_tidepool ok 10
kelp_tank ok 25
pier_cylinder ok 5
jelly_tumbler ok 1
tank_25 ok 3
tank_4 ok 12
jelly_lab_sump ok 2
tank_7 ok 14
tank_5 ok 23
tank_6 ok 8
boreal_3 ok 1
tank_3 ok 19
boreal_1 ok 2
boreal_2 ok 6
or_6 no organism data (expected)
or_7 ok 3
tank_17 ok 2
pier_piling ok 9
tank_18 ok 10
tank_16 ok 8
tank_2 ok 5
bts_gpo no organism data (expected)
display_gpo ok 9
tank_15 ok 16
shark_shores ok 9
or_1 ok 2
or_2 ok 1
or_3 ok 18
or_4 ok 1
or_5 ok 8
sd_1 ok 1
